# Paper Disruption Trend — year-by-year disruption (CD) and citer profile (F/E/G) per paper

For every paper with at least one citer, its **disruption index and citer profile as they evolve**,
anchored at the **publication year**: one row per paper × year in which anything changed, carrying

- `ni_new, nj_new, nk_new` — citers / co-citing papers that appeared **in that year**
  ($n_i$: cites the paper but none of its references; $n_j$: cites the paper **and** ≥ 1 of its
  references; $n_k$: cites ≥ 1 of its references but not the paper);
- `ni, nj, nk` — the **cumulative** counts from the publication year through that year;
- `CD` — $\dfrac{n_i-n_j}{n_i+n_j+n_k}$ on the cumulative counts, i.e. the disruption index with a
  citation window of exactly `yrs_since_pub` years;
- `F, E, G` — the Foundational / Extensional / Generalizational shares of the citers that have
  arrived by that year (`up` = references shared with the paper, `down` = citers of the paper the
  citer also cites, both counted within the window), NaN until the first citer.

This is the time axis that `paper_disruption.ipynb` collapses into four windows: the row at
`yrs_since_pub = w` (or the last row before it) **is** `CD_w` / `F_w` / `E_w` / `G_w`, and the last
row is the `_all` value. Section 3 checks that identity on **every** focal paper against
`paper_disruption.parquet`. Same relationship as `paper_citation_trend` has to `paper_citation`.

## Raw / input data
```
/project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_csr.npz        # int32 CSR of the OpenAlex citation graph (built by paper_disruption): out/in adjacency, year, uni_mag
/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_disruption.parquet   # per-window CD/F/E/G/ni/nj/nk; the check target
```
Codes are positions in `uni_mag` (sorted accession numbers); `paper_id = W{code}`.

## Engine
The `paper_disruption` numba kernel, re-cut along time: each focal paper's citers are classified
i / j once (a citer's type does not depend on the window) and binned by citing year; the distinct
citers of its references are binned by their year; prefix sums over years give the cumulative
$n_i, n_j, n_k$ and CD. F/E/G do depend on the window (`down` counts co-cited citers inside it), so
each citer's `down` is re-counted at every age it is present. Focal papers are processed in
batches and streamed to parquet, because the result is ~1.2 B rows.

## Output
`/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_disruption_trend.parquet` —
`paper_id, pub_year, cite_year, yrs_since_pub, ni_new, nj_new, nk_new, ni, nj, nk, CD, F, E, G`
(one row per paper × year with ≥ 1 new citer or co-citing paper; `CD, F, E, G` are cumulative to that year).

`/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_disruption_trend_summary.parquet` —
`pub_year, yrs_since_pub, n_<m>, <m>_mean` for `m` in CD/F/E/G: mean cumulative outcomes by cohort and age.

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import numba
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa
ROOT = oa.BASE; OUT = oa.OUT
print('snapshot:', oa.ROOT)
OUT_FP  = f'{OUT}/paper_disruption_trend.parquet'
SUM_FP  = f'{OUT}/paper_disruption_trend_summary.parquet'
DISR_FP = f'{OUT}/paper_disruption.parquet'     # per-window table: the check target in §3
BATCH   = 2_000_000                              # focal papers per kernel call / write
# numba otherwise sizes its pool from the node's core count, not the job's allocation.
NT = int(os.environ.get('SLURM_CPUS_PER_TASK', numba.config.NUMBA_NUM_THREADS))
numba.set_num_threads(min(NT, numba.config.NUMBA_NUM_THREADS))
print('numba threads:', numba.get_num_threads())
# Everything below is fed by notebook/referenced_works_w_year.ipynb: it writes the per-work
# map (year + source) and the edge table with both years and both source ids, and
# oa.load_graph() / oa.load_csr() / oa.build_journal() read those instead of re-walking
# renli's tree. Build it once before running this notebook.
assert oa.have_consolidated(), (
    'run notebook/referenced_works_w_year.ipynb first — it builds the map and edge table')
oa.summary()

snapshot: /project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet
numba threads: 32
snapshot : /project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet
cache    : /project/jevans/Dawoon/Science of Science/OpenAlex/cache
output   : /project/jevans/Dawoon/Science of Science/OpenAlex/output
  consolidated edge table: present
  map            6.71 GB  /project/jevans/Dawoon/Science of Science/OpenAlex/cache/work_year_source_map.npz
  graph         21.62 GB  /project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_graph.npz
  csr           27.20 GB  /project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_csr.npz
  journal        3.44 GB  /project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_journal.parquet
  fos            2.62 GB  /project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_fos.parquet


## 1. Load int32 CSR (built and cached by paper_disruption)

In [2]:
%%time
out_ptr, out_idx, in_ptr, in_idx, year, uni_mag = oa.load_csr()   # builds on first run
n = len(year); YMAX = int(year.max())
print(f'CSR: {n:,} papers, {len(out_idx):,} edges | years {int(year.min())}..{YMAX}')

CSR cache present: /project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_csr.npz
CSR: 348,939,837 papers, 2,178,823,795 edges | years 1000..2030


## 2. Numba engine — yearly n_i / n_j / n_k and F / E / G per focal paper, then prefix sums

In [3]:
from numba import njit, prange

@njit(inline='always')
def _bfind(arr, x):
    lo = 0; hi = len(arr)
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid] < x: lo = mid + 1
        else: hi = mid
    return lo < len(arr) and arr[lo] == x

@njit(parallel=True)
def trend_kernel(focal, off, out_ptr, out_idx, in_ptr, in_idx, year,
                 ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf):
    """Per focal document, per year since publication: new and cumulative n_i / n_j / n_k, CD,
    and the cumulative citer profile F / E / G.

    Dense layout: focal t owns slots off[t] .. off[t+1]-1, slot off[t]+d being year yF+d.
    ni/nj/nk receive the yearly increments, NI/NJ/NK the running totals, CD the running index,
    Ff/Ef/Gf the shares of the citers of age <= d that are Foundational / Extensional /
    Generalizational at age d. Slots of different focal documents never overlap, so the prange
    writes are race-free.

    Same definitions as the per-window engine (paper_disruption / patent_disruption):
      n_j(y) = citers published in year y that cite >= 1 reference of the focal;
      n_i(y) = the other citers of year y;
      n_k(y) = |B_y| - n_j(y), B_y = distinct documents of year y citing >= 1 reference of the
               focal, the focal itself excluded;
      citer c: up = |refs(F) & refs(c)|, down(d) = |{citers of F of age <= d} & refs(c)|;
               up > down -> E, down > up -> F, up = down = 0 -> G, up = down > 0 -> half F, half E.
    Summing over years 0..w reproduces ni_w / nj_w / nk_w / F_w / E_w / G_w exactly, because a
    window in the per-window engine is precisely a prefix of years. `down` grows with the age,
    which is why the profile is re-cut at every age instead of once.
    """
    for t in prange(len(focal)):
        F = focal[t]; yF = year[F]; b0 = off[t]; b1 = off[t + 1]; span = b1 - b0
        a0 = in_ptr[F]; a1 = in_ptr[F + 1]
        R = out_idx[out_ptr[F]:out_ptr[F + 1]]
        Rs = np.sort(R); As = np.sort(in_idx[a0:a1])
        # per-age citer-profile counters; a citer of age dc is present at every age >= dc
        Nc = np.zeros(span, np.int64); cF = np.zeros(span, np.int64); cE = np.zeros(span, np.int64)
        cG = np.zeros(span, np.int64); cT = np.zeros(span, np.int64)
        maxdeg = 0
        for ci in range(a0, a1):
            c = in_idx[ci]; deg = out_ptr[c + 1] - out_ptr[c]
            if deg > maxdeg: maxdeg = deg
        dds = np.empty(maxdeg, np.int64)
        for ci in range(a0, a1):
            c = in_idx[ci]; dc = year[c] - yF
            if dc < 0:
                continue
            up = 0; nd = 0
            for ri in range(out_ptr[c], out_ptr[c + 1]):
                d = out_idx[ri]
                if _bfind(Rs, d): up += 1
                if _bfind(As, d):
                    dd = year[d] - yF
                    if dd >= 0:
                        dds[nd] = dd; nd += 1
            if up > 0: nj[b0 + dc] += 1
            else:      ni[b0 + dc] += 1
            dsort = np.sort(dds[:nd]); p = 0
            for tt in range(dc, span):
                while p < nd and dsort[p] <= tt: p += 1     # down(tt) = co-cited citers of age <= tt
                Nc[tt] += 1
                if up > p:   cE[tt] += 1
                elif p > up: cF[tt] += 1
                elif up > 0: cT[tt] += 1
                else:        cG[tt] += 1
        # B: distinct citers of the focal's references (focal excluded), by year
        totB = 0
        for ri in range(len(R)):
            r = R[ri]; totB += in_ptr[r + 1] - in_ptr[r]
        if totB > 0:
            buf = np.empty(totB, np.int32); p = 0
            for ri in range(len(R)):
                r = R[ri]
                for j in range(in_ptr[r], in_ptr[r + 1]):
                    buf[p] = in_idx[j]; p += 1
            buf.sort(); prev = np.int32(-1)
            for ii in range(totB):
                b = buf[ii]
                if b == prev or b == F:
                    continue
                prev = b; dd = year[b] - yF
                if dd >= 0:
                    nk[b0 + dd] += 1
        # nk holds |B_y| so far; turn it into n_k(y) = |B_y| - n_j(y), then cumulate
        si = 0; sj = 0; sk = 0
        for q in range(span):
            p = b0 + q
            nk[p] -= nj[p]
            si += ni[p]; sj += nj[p]; sk += nk[p]
            NI[p] = si; NJ[p] = sj; NK[p] = sk
            den = si + sj + sk
            CD[p] = (si - sj) / den if den > 0 else np.nan
            if Nc[q] > 0:
                Ff[p] = (cF[q] + 0.5 * cT[q]) / Nc[q]
                Ef[p] = (cE[q] + 0.5 * cT[q]) / Nc[q]
                Gf[p] = cG[q] / Nc[q]
            else:
                Ff[p] = np.nan; Ef[p] = np.nan; Gf[p] = np.nan
print('engine ready')

engine ready


## 3. Run over all focal papers (with citers) in batches, stream to parquet, check every one against paper_disruption.parquet

In [4]:
%%time
focal_all = np.flatnonzero(in_ptr[1:] - in_ptr[:-1] > 0).astype(np.int64)
print(f'focal papers with >= 1 citer: {len(focal_all):,}  (batches of {BATCH:,})')
YMIN = int(year.min()); NY = YMAX - YMIN + 1
OUTC = ['CD', 'F', 'E', 'G']; CNTS = ['ni', 'nj', 'nk']
SCHEMA = pa.schema([('paper_id', pa.string()), ('pub_year', pa.int32()), ('cite_year', pa.int32()),
                    ('yrs_since_pub', pa.int32()),
                    ('ni_new', pa.int32()), ('nj_new', pa.int32()), ('nk_new', pa.int32()),
                    ('ni', pa.int32()), ('nj', pa.int32()), ('nk', pa.int32()),
                    ('CD', pa.float32()), ('F', pa.float32()), ('E', pa.float32()), ('G', pa.float32())])
# (pub_year, yrs_since_pub) accumulators for the summary in the next section; filled per batch so the
# summary never has to re-read the (large) output.
sum_o = {m: np.zeros(NY * NY, np.float64) for m in OUTC}; cnt_o = {m: np.zeros(NY * NY, np.int64) for m in OUTC}

# ── the per-window table, for the exhaustive check. Its rows are in code order (one row per
#    entry of the id array, written in that order), so row i is document i; the id column is
#    checked against the id array before anything is compared.
WIN = {'_3': 3, '_5': 5, '_10': 10, '_all': None}
_dt = pq.read_table(DISR_FP)
assert (np.asarray(_dt.column('paper_id').to_pandas()) == oa.code_to_id(uni_mag)).all(), 'per-window table is not in code order'
DIS = {}
for s_ in WIN:
    for m in OUTC:
        DIS[m + s_] = _dt.column(m + s_).to_numpy(zero_copy_only=False).astype(np.float32)
    for m in CNTS:                     # -1 (paper) or NaN (patent) both mean "nothing in the window"
        v = _dt.column(m + s_).to_numpy(zero_copy_only=False).astype(np.float64)
        DIS[m + s_] = np.where(np.isfinite(v), v, -1).astype(np.int64)
del _dt; gc.collect()
print(f'per-window table loaded for the check: {len(DIS["CD_3"]):,} rows x {len(DIS)} columns')
mism = {s_: {m: 0 for m in OUTC + CNTS + ['undefined']} for s_ in WIN}
ncmp = {s_: 0 for s_ in WIN}

TMP = OUT_FP + '.tmp'
writer = pq.ParquetWriter(TMP, SCHEMA, compression='zstd')
rows_total = 0; docs_total = 0; t0 = time.time()
nb = (len(focal_all) + BATCH - 1) // BATCH
for bi, s in enumerate(range(0, len(focal_all), BATCH)):
    focal = focal_all[s:s + BATCH]
    yF = year[focal].astype(np.int64)
    span = YMAX - yF + 1
    off = np.zeros(len(focal) + 1, np.int64); np.cumsum(span, out=off[1:])
    T = int(off[-1])
    ni = np.zeros(T, np.int32); nj = np.zeros(T, np.int32); nk = np.zeros(T, np.int32)
    NI = np.empty(T, np.int32); NJ = np.empty(T, np.int32); NK = np.empty(T, np.int32)
    CD = np.empty(T, np.float32); Ff = np.empty(T, np.float32); Ef = np.empty(T, np.float32); Gf = np.empty(T, np.float32)
    trend_kernel(focal, off, out_ptr, out_idx, in_ptr, in_idx, year, ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf)

    # ── exhaustive check: the cumulative slot at age w IS the per-window value at w. When the
    #    document's span ends before w the last slot holds the total, which is also the window.
    OUT_ARR = {'CD': CD, 'F': Ff, 'E': Ef, 'G': Gf, 'ni': NI, 'nj': NJ, 'nk': NK}
    for s_, w in WIN.items():
        slot = off[:-1] + (span - 1 if w is None else np.minimum(w, span - 1))
        undefined = (NI[slot] == 0) & (NJ[slot] == 0) & (NK[slot] == 0)      # nothing in the window
        ref_undef = DIS['ni' + s_][focal] == -1
        mism[s_]['undefined'] += int((undefined != ref_undef).sum())
        ok = ~undefined & ~ref_undef
        for m in CNTS:
            mism[s_][m] += int((OUT_ARR[m][slot][ok] != DIS[m + s_][focal][ok]).sum())
        for m in OUTC:
            mism[s_][m] += int((~np.isclose(OUT_ARR[m][slot][ok], DIS[m + s_][focal][ok], atol=1e-6, equal_nan=True)).sum())
        ncmp[s_] += int(ok.sum())

    # one row per (paper, year) with any event: a new citer, or a new co-citing document
    ridx = np.flatnonzero((ni != 0) | (nj != 0) | (nk != 0))
    fpos = np.searchsorted(off, ridx, side='right') - 1          # focal position of each row
    yrs = (ridx - off[fpos]).astype(np.int32)
    pub = yF[fpos].astype(np.int32)
    ids = pa.array(oa.code_to_id(uni_mag[focal]))
    tbl = pa.table({'paper_id': ids.take(pa.array(fpos)), 'pub_year': pub, 'cite_year': pub + yrs, 'yrs_since_pub': yrs,
                    'ni_new': ni[ridx], 'nj_new': nj[ridx], 'nk_new': nk[ridx],
                    'ni': NI[ridx], 'nj': NJ[ridx], 'nk': NK[ridx],
                    'CD': CD[ridx], 'F': Ff[ridx], 'E': Ef[ridx], 'G': Gf[ridx]}, schema=SCHEMA)
    writer.write_table(tbl)
    # CD is NaN on the rare row whose denominator is 0 (a self-citing document), F/E/G are NaN
    # until the first citer arrives; the means are over defined values only, as in the by-year notebooks.
    key_all = (pub.astype(np.int64) - YMIN) * NY + yrs
    for m in OUTC:
        v = OUT_ARR[m][ridx]; fin = np.isfinite(v)
        sum_o[m] += np.bincount(key_all[fin], weights=v[fin].astype(np.float64), minlength=NY * NY)
        cnt_o[m] += np.bincount(key_all[fin], minlength=NY * NY)
    rows_total += len(ridx); docs_total += int((np.bincount(fpos, minlength=len(focal)) > 0).sum())
    print(f'[{time.time()-t0:6.0f}s] batch {bi+1:>3}/{nb}: {len(focal):,} focal -> '
          f'{len(ridx):,} rows  (cum {rows_total:,})', flush=True)
    del ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf, OUT_ARR, ridx, fpos, yrs, pub, ids, tbl, key_all, v, fin; gc.collect()
writer.close()
os.replace(TMP, OUT_FP)
del DIS; gc.collect()
print(f'WROTE {OUT_FP}  ({rows_total:,} rows, {os.path.getsize(OUT_FP)/1e9:.1f} GB)')
print(f'  papers covered: {docs_total:,} | cite_year {YMIN}..{YMAX}')

focal papers with >= 1 citer: 89,073,501  (batches of 2,000,000)
per-window table loaded for the check: 348,939,837 rows x 28 columns
[   125s] batch   1/45: 2,000,000 focal -> 34,361,130 rows  (cum 34,361,130)
[   214s] batch   2/45: 2,000,000 focal -> 21,736,617 rows  (cum 56,097,747)
[   356s] batch   3/45: 2,000,000 focal -> 32,330,723 rows  (cum 88,428,470)
[   513s] batch   4/45: 2,000,000 focal -> 33,813,671 rows  (cum 122,242,141)
[   684s] batch   5/45: 2,000,000 focal -> 38,927,444 rows  (cum 161,169,585)
[   840s] batch   6/45: 2,000,000 focal -> 45,688,655 rows  (cum 206,858,240)
[   996s] batch   7/45: 2,000,000 focal -> 45,720,683 rows  (cum 252,578,923)
[  1153s] batch   8/45: 2,000,000 focal -> 45,685,580 rows  (cum 298,264,503)
[  1308s] batch   9/45: 2,000,000 focal -> 45,721,609 rows  (cum 343,986,112)
[  1466s] batch  10/45: 2,000,000 focal -> 45,768,611 rows  (cum 389,754,723)
[  1623s] batch  11/45: 2,000,000 focal -> 45,748,388 rows  (cum 435,503,111)
[  1780s] b

In [5]:
# Every focal paper (not a sample), every window, every metric: the cumulative slot at age w
# against paper_disruption.parquet. `undefined` counts documents where one side says "nothing in the window"
# and the other does not.
print(f'{"window":<8}{"compared":>14}' + ''.join(f'{m:>10}' for m in OUTC + CNTS) + f'{"undefined":>12}')
print('-' * 92)
total_bad = 0
for s_ in WIN:
    print(f'{s_:<8}{ncmp[s_]:>14,}' + ''.join(f'{mism[s_][m]:>10,}' for m in OUTC + CNTS) + f'{mism[s_]["undefined"]:>12,}')
    total_bad += sum(mism[s_].values())
assert total_bad == 0, 'trend disagrees with the per-window table'
print(f'\n  cumulative trend == per-window CD / F / E / G / ni / nj / nk at every window, on all {len(focal_all):,} papers')

window        compared        CD         F         E         G        ni        nj        nk   undefined
--------------------------------------------------------------------------------------------
_3          76,433,207         0         0         0         0         0         0         0           0
_5          79,840,156         0         0         0         0         0         0         0           0
_10         83,659,116         0         0         0         0         0         0         0           0
_all        88,530,724         0         0         0         0         0         0         0           0

  cumulative trend == per-window CD / F / E / G / ni / nj / nk at every window, on all 89,073,501 papers


## 4. Summary — mean cumulative CD / F / E / G by age, pooled and by cohort; head of the output

In [6]:
cells = np.flatnonzero(sum(cnt_o.values()))
py, yy = np.divmod(cells, NY)
summ = pd.DataFrame({'pub_year': py + YMIN, 'yrs_since_pub': yy})
with np.errstate(invalid='ignore', divide='ignore'):
    for m in OUTC:
        summ[f'n_{m}'] = cnt_o[m][cells]
        summ[f'{m}_mean'] = sum_o[m][cells] / cnt_o[m][cells]
summ.to_parquet(SUM_FP, index=False)
print(f'WROTE {SUM_FP}  ({len(summ):,} (pub_year, yrs_since_pub) cells)')

# Mean cumulative CD / F / E / G by years since the paper year, all papers pooled. Reading down a
# column is how each outcome moves with the citation window.
S = {m: sum_o[m].reshape(NY, NY) for m in OUTC}; C = {m: cnt_o[m].reshape(NY, NY) for m in OUTC}
YRS = [0, 1, 2, 3, 5, 10, 15, 20, 30]
pooled = pd.DataFrame({'yrs_since_pub': YRS, 'n_CD': [int(C['CD'][:, y].sum()) for y in YRS], 'n_FEG': [int(C['F'][:, y].sum()) for y in YRS],
                       **{f'{m}_mean': [S[m][:, y].sum() / max(C[m][:, y].sum(), 1) for y in YRS] for m in OUTC}})
display(pooled.round(4))

# By cohort: each outcome at fixed ages, so cohorts are compared at the same window length.
rows = []
for lo, hi in ((1950, 1959), (1960, 1969), (1970, 1979), (1980, 1989), (1990, 1999), (2000, 2009), (2010, 2019)):
    if hi < YMIN or lo > YMAX:
        continue
    r = slice(max(lo, YMIN) - YMIN, hi - YMIN + 1)
    rows.append({'cohort': f'{lo}-{hi}', 'n_rows': int(C['CD'][r].sum()),
                 **{f'{m}@{y}': (S[m][r, y].sum() / C[m][r, y].sum() if C[m][r, y].sum() else np.nan)
                    for m in OUTC for y in (1, 3, 5, 10, 20)}})
COH = pd.DataFrame(rows).set_index('cohort')
for m in OUTC:
    print(f'\n{m} by cohort at fixed ages:')
    display(COH[[f'{m}@{y}' for y in (1, 3, 5, 10, 20)]].round(4))

head = pq.ParquetFile(OUT_FP).read_row_group(0).to_pandas()
with pd.option_context('display.width', 250, 'display.max_columns', 20):
    display(head.head(12))

WROTE /project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_disruption_trend_summary.parquet  (34,282 (pub_year, yrs_since_pub) cells)


,yrs_since_pub,n_CD,n_FEG,CD_mean,F_mean,E_mean,G_mean
0,0,64300145,21150390,0.0463,0.0189,0.5916,0.3895
1,1,67513013,46354532,0.0926,0.0268,0.5722,0.4010
2,2,67648750,55481397,0.1055,0.0435,0.5552,0.4013
3,3,65787369,57768471,0.1064,0.0613,0.5422,0.3965
4,5,59440312,55123307,0.1029,0.0947,0.5207,0.3846
5,10,42905007,41368030,0.0976,0.1571,0.4811,0.3619
6,15,30468682,29718713,0.1002,0.1996,0.4495,0.3509
7,20,21495010,21070544,0.1076,0.2308,0.4225,0.3466
8,30,10961970,10806476,0.1150,0.2552,0.4005,0.3444



CD by cohort at fixed ages:


,CD@1,CD@3,CD@5,CD@10,CD@20
cohort,,,,,
1950-1959,0.2369,0.2605,0.2433,0.2183,0.1849
1960-1969,0.1893,0.2162,0.2029,0.1739,0.1415
1970-1979,0.1613,0.1862,0.1747,0.1461,0.1124
1980-1989,0.1432,0.1744,0.1635,0.1372,0.1266
1990-1999,0.1355,0.1692,0.1663,0.1546,0.1345
2000-2009,0.0788,0.0965,0.0960,0.0880,0.0576
2010-2019,0.0747,0.0896,0.0790,0.0490,-0.0002



F by cohort at fixed ages:


...


E by cohort at fixed ages:


,E@1,E@3,E@5,E@10,E@20
cohort,,,,,
1950-1959,0.3600,0.3662,0.3693,0.3607,0.3457
1960-1969,0.4421,0.4293,0.4226,0.4064,0.3873
1970-1979,0.4958,0.4833,0.4728,0.4533,0.4330
1980-1989,0.5401,0.5179,0.5056,0.4812,0.4363
1990-1999,0.5516,0.5255,0.5045,0.4597,0.4092
2000-2009,0.6039,0.5687,0.5389,0.4845,0.4373
2010-2019,0.5980,0.5518,0.5260,0.5032,0.5159



G by cohort at fixed ages:


,G@1,G@3,G@5,G@10,G@20
cohort,,,,,
1950-1959,0.6026,0.5501,0.5112,0.4656,0.4301
1960-1969,0.5183,0.4814,0.4492,0.4086,0.3784
1970-1979,0.4709,0.4377,0.4097,0.3715,0.3433
1980-1989,0.4316,0.4119,0.3872,0.3540,0.3419
1990-1999,0.4213,0.4073,0.3894,0.3649,0.3472
2000-2009,0.3709,0.3703,0.3650,0.3571,0.3347
2010-2019,0.3766,0.3880,0.3836,0.3555,0.3393


,paper_id,pub_year,cite_year,yrs_since_pub,ni_new,nj_new,nk_new,ni,nj,nk,CD,F,E,G
0,W23,2012,2018,6,1,0,0,1,0,0,1.000000,0.0,0.0,1.0
1,W23,2012,2019,7,2,0,0,3,0,0,1.000000,0.0,0.0,1.0
2,W23,2012,2020,8,1,0,0,4,0,0,1.000000,0.0,0.0,1.0
3,W23,2012,2022,10,1,0,0,5,0,0,1.000000,0.0,0.0,1.0
4,W125,1988,1988,0,1,0,464,1,0,464,0.002151,0.0,0.0,1.0
5,W125,1988,1989,1,0,0,415,1,0,879,0.001136,0.0,0.0,1.0
6,W125,1988,1990,2,0,0,398,1,0,1277,0.000782,0.0,0.0,1.0
7,W125,1988,1991,3,0,0,428,1,0,1705,0.000586,0.0,0.0,1.0
8,W125,1988,1992,4,0,0,326,1,0,2031,0.000492,0.0,0.0,1.0
9,W125,1988,1993,5,0,0,337,1,0,2368,0.000422,0.0,0.0,1.0
